# LLM-as-a-Judge Inference

We run an instruction-tuned LLM to judge answer quality.
Multiple samples per prompt are generated for robustness.


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from inference_helpers import (
    load_disk_records,
    load_arrow_records,
    build_prompt_records,
    run_best_of_n,
    run_batched_inference,
)


In [ ]:
import os
from pathlib import Path


MODE = "best_of_n"   # options: "best_of_n" | "batched"
MODEL_ID = "Qwen/Qwen2-7B-Instruct"

# Resolve repo root
REPO_ROOT = Path(__file__).resolve().parents[1]

# Paths
HF_DATASET_PATH = REPO_ROOT / "Sky_dataset"
ARROW_DATASET_PATH = (
    HF_DATASET_PATH / "train" / "data-00000-of-00001.arrow"
)

OUT_DIR  = REPO_ROOT / "Sky_outputs"
CKPT_DIR = OUT_DIR / "checkpoints"

# Create dirs
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

# Model
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
).eval()

print("Model loaded:", MODEL_ID)
print("Attention backend:", model.config._attn_implementation)


In [ ]:
if MODE == "best_of_n":

    records = load_disk_records(HF_DATASET_PATH, limit=200)

    run_best_of_n(
        records=records,
        model=model,
        tokenizer=tokenizer,
        output_path=os.path.join(OUT_DIR, "best_of_n.jsonl"),
        checkpoint_dir=os.path.join(CKPT_DIR, "best_of_n"),
        task_name="best_of_n",
        n=8,
        checkpoint_every=5,
        max_new_tokens=512,
    )

    print("Best-of-N inference completed")


In [ ]:
if MODE == "batched":

    templates = {
        "guide": [
            "As an evaluation expert, given a question and its two possible answers, "
            "please analyze the performance of both in terms of coherence, accuracy, "
            "coverage, and the overall quality defined above.\nQuestion:",
            "\nAnswer 1:",
            "\nAnswer 2:",
            "\nKnown answer ",
            " is better, please explain the reason:",
        ],
        "guide_reverse": [
            "As an evaluation expert, given a question and its two possible answers, "
            "please analyze the performance of both in terms of coherence, accuracy, "
            "coverage, and the overall quality defined above.\nQuestion:",
            "\nAnswer 1:",
            "\nAnswer 2:",
            "\nKnown answer ",
            " is better, please explain the reason:",
        ],
    }

    raw = load_arrow_records(ARROW_DATASET_PATH, limit=200)

    batched_outputs = {}

    for key in ["guide", "guide_reverse"]:
        print(f"\n▶ Running batched inference for: {key}")

        records = build_prompt_records(
            dataset=raw,
            templates=templates,
            template_key=key,
            reverse=(key == "guide_reverse"),
        )

        batched_outputs[key] = run_batched_inference(
            records=records,
            model=model,
            tokenizer=tokenizer,
            batch_size=4,
            max_new_tokens=512,
        )

        # 🔍 Inspect a few samples (CORRECT)
        for r in batched_outputs[key][:5]:
            print(f"[{key}] {r['prompt_idx']} → {r['output'][:120]}")


In [ ]:
import jsonlines

if MODE == "batched":

    for key, results in batched_outputs.items():
        out_path = os.path.join(OUT_DIR, f"{key}.jsonl")

        with jsonlines.open(out_path, "w") as writer:
            for r in results:
                writer.write(r)

        print(f"💾 Saved → {out_path}")